# Model Validation and Grid Convergence

This notebook validates two aspects of the subsidence implementation:

1. the integrated surface-volume response of the Geertsma and Van Opstal influence functions;
2. convergence of the synthetic spatial model as the reservoir source grid is refined.

These checks complement the unit tests by validating physical integral behaviour and numerical discretization.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from reservoir_subsidence.validation import (
    bowl_volume_convergence,
    source_grid_convergence,
)

## 1. Integrated Subsidence-Bowl Volume

For a unit point-compaction source, the dimensionless ratio

```math
\frac{V_{\mathrm{subs}}}{V_{\mathrm{comp}}}
=
\int_0^{R_{\max}} 2\pi r\,G(r)\,dr
```

can be evaluated numerically.

For the homogeneous Geertsma half-space solution, the infinite-domain analytical result is

```math
\frac{V_{\mathrm{subs}}}{V_{\mathrm{comp}}}
=
2(1-\nu)
```

which equals **1.5** for $\nu=0.25$.

With the Van Opstal rigid-basement correction, the integrated response should approach **1.0**, corresponding to volume conservation.

In [ ]:
volume_validation = bowl_volume_convergence(
    reservoir_depth_m=2000.0,
    basement_depth_m=2200.0,
    radius_multipliers=(5, 10, 20, 50, 100, 250),
    n_radial_points=5000,
)

volume_validation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    volume_validation["integration_radius_over_depth"],
    volume_validation["geertsma_volume_ratio"],
    marker="o",
    label="Geertsma",
)
ax.plot(
    volume_validation["integration_radius_over_depth"],
    volume_validation["van_opstal_volume_ratio"],
    marker="o",
    label="Van Opstal",
)

ax.axhline(
    1.5,
    linestyle="--",
    linewidth=1,
    label="Geertsma analytical limit",
)
ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
    label="Volume-conservation limit",
)

ax.set_xscale("log")
ax.set_xlabel("Integration Radius / Reservoir Depth [-]")
ax.set_ylabel("Integrated Bowl Volume / Compaction Volume [-]")
ax.set_title("Influence-Function Volume Validation")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plt.show()

At an integration radius of 250 reservoir depths, the numerical results are approximately:

- Geertsma: **1.494**, approaching the analytical target of **1.5**;
- Van Opstal: **1.0007**, approaching volume conservation.

The residual Geertsma difference is primarily the finite integration-domain truncation of the long-range elastic tail.

## 2. Source-Grid Convergence

The same synthetic two-reservoir model is evaluated using source-cell spacings of:

- 400 m
- 200 m
- 100 m

The surface evaluation grid is held fixed at 200 m so that the comparison isolates the effect of reservoir source discretization.

The following quantities are monitored:

- total number of source cells;
- total discretized compaction volume;
- maximum predicted subsidence;
- subsidence at $(0,0)$;
- integrated subsidence-bowl volume.

In [ ]:
grid_convergence = source_grid_convergence(
    source_spacings_m=(400.0, 200.0, 100.0),
    surface_spacing_m=200.0,
    observation_x_m=0.0,
    observation_y_m=0.0,
)

grid_convergence

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    grid_convergence["source_spacing_m"],
    grid_convergence["maximum_subsidence_mm"],
    marker="o",
    label="Maximum subsidence",
)
ax.plot(
    grid_convergence["source_spacing_m"],
    grid_convergence["observation_subsidence_mm"],
    marker="o",
    label="Subsidence at (0, 0)",
)

ax.invert_xaxis()
ax.set_xlabel("Source-Cell Spacing [m]")
ax.set_ylabel("Subsidence [mm]")
ax.set_title("Source-Grid Convergence")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plt.show()

Using the 100 m source grid as the finest reference:

- 400 m source spacing differs by about **6.0%** in peak subsidence;
- 200 m source spacing differs by about **1.6%**;
- 100 m is the reference case.

The convergence study also captures changes in discretized reservoir area and total compaction as the elliptical synthetic reservoir boundaries are rasterized more finely. It therefore tests the complete source-discretization workflow rather than only the point-source kernel.

## 3. Interpretation

These tests validate two different aspects of the implementation:

**Physical integral behaviour**
- the Geertsma solution approaches its analytical $2(1-\nu)$ bowl-volume ratio;
- the Van Opstal rigid-basement solution approaches volume conservation.

**Numerical discretization**
- predicted subsidence stabilizes as the source grid is refined;
- the 200 m source grid is already close to the 100 m reference for this synthetic geometry.

Grid-convergence tolerances remain application-specific. Field studies should repeat the exercise using the actual reservoir geometry and the quantities relevant to the decision being supported.